# String Operations in Pandas

**Learning Objectives:**
- Understand how to work with text data using the `.str` accessor
- Apply common string methods like `.upper()`, `.lower()`, `.split()`, and `.replace()`
- Extract patterns from text using regular expressions
- Combine string columns using different techniques
- Clean and transform real-world text data

[String methods documentation](https://pandas.pydata.org/docs/user_guide/text.html)

In [ ]:
import pandas as pd

---

## 1. Introduction to string operations

📚 **Concept**: When working with text data in pandas, we need special methods to manipulate strings. Pandas provides a powerful **`.str` accessor** that unlocks string methods for Series objects.

Think of `.str` as a gateway to Python's string methods, adapted to work on entire columns at once.

<!-- INSTRUCTOR: Emphasize that string operations work on entire columns, not individual values -->

### 1.1 Sample data

Let's create a sample dataset to explore string operations:

In [ ]:
# Create sample data with city/state and number codes
sample = {
    'col_a': ['Houston,TX', 'Dallas,TX', 'Chicago,IL', 'Phoenix,AZ', 'San Diego,CA'],
    'col_b': ['no: (1511)', 'no: (007)', 'no: (333)', 'no: (135)', 'no: (7833)'],
    'col_c': ['A', 'B', 'C', 'D', 'E'],
    'col_d': ['a', 'b', 'c', 'd', 'e']
}

df = pd.DataFrame(sample)
df

---

## 2. The `.str` accessor

📚 **Concept**: In pandas, Series (columns) don't have string methods directly. We need to access them through the **`.str` accessor**.

Syntax: `series.str.string_method()`

<!-- INSTRUCTOR: Show the error first, then the correct way. This helps students understand WHY we use .str -->

### 2.1 Why we need `.str`

Let's see what happens if we try to use a string method without `.str`:

In [ ]:
# ⚠️ INTENTIONAL ERROR - This will fail!
# df["col_a"].upper()
# Error: AttributeError - 'Series' object has no attribute 'upper'

⚠️ **Why this fails**: The `.upper()` method exists for individual strings, but not for Series objects.

✅ **Solution**: Use the `.str` accessor first!

In [ ]:
# Correct way: use .str accessor
df["col_a"].str.upper()

💡 **Tip**: Think of `.str` as unlocking string operations for the entire column. Each value in the Series gets the method applied to it.

---

## 3. Common string methods

📚 **Concept**: Pandas provides many string methods through the `.str` accessor. Let's explore the most common ones:

- `.upper()` / `.lower()` - Change case
- `.split()` - Split strings into parts
- `.replace()` - Replace characters or patterns

<!-- INSTRUCTOR: Build from simple examples to more complex ones -->

### 3.1 Changing case

In [ ]:
# Convert to uppercase
df["col_a"].str.upper()

In [ ]:
# Convert to lowercase
df["col_a"].str.lower()

### 3.2 Splitting strings

📚 **Concept**: The `.split()` method breaks strings into parts based on a delimiter (separator character).

When we use `expand=True`, the result becomes a DataFrame with separate columns for each part.

In [ ]:
# Split on comma, expand into separate columns
df["col_a"].str.split(",", expand=True)

💡 **Understanding the output**: 
- Column `0` contains the city names
- Column `1` contains the state codes
- `expand=True` creates a DataFrame instead of a Series of lists

### 3.3 Selecting parts after splitting

We can select individual columns from the split result:

In [ ]:
# Get just the city names (first part)
df["col_a"].str.split(",", expand=True)[0]

In [ ]:
# Get just the state codes (second part)
df["col_a"].str.split(",", expand=True)[1]

### 3.4 Creating new columns from split data

💡 **Pro Tip**: When splitting into multiple columns, assign them all at once to avoid repeating the split operation:

In [ ]:
# Efficient way: assign both columns at once
df[["City", "State"]] = df["col_a"].str.split(",", expand=True)
df

✅ **Key Takeaway**: This is more efficient than splitting twice, and the code is cleaner!

---

## 4. Extracting patterns with regex

📚 **Concept**: **Regular expressions (regex)** are patterns used to match specific text. The `.extract()` method uses regex to pull out parts of strings.

Common regex patterns:
- `\d` - matches any digit (0-9)
- `\d+` - matches one or more digits
- `()` - capturing group (what to extract)

[Regex tutorial](https://regexone.com/)

<!-- INSTRUCTOR: Keep regex simple for beginners. Only show what's needed for this example -->

### 4.1 Extracting numbers from text

Let's extract the numbers from `col_b` which contains values like `"no: (1511)"`:

In [ ]:
# Extract one or more digits using regex
df["col_b"].str.extract(r'(\d+)')
# Pattern breakdown:
# \d+ means "one or more digits"
# () means "capture this part"

💡 **Tip**: The `r` before the string makes it a "raw string" which is needed for regex patterns.

In [ ]:
# Create a new column with the extracted codes
df["code"] = df["col_b"].str.extract(r'(\d+)')
df

⚠️ **Note**: Regex can be complex! For this course, we'll only use basic patterns. Feel free to explore more advanced regex as you gain experience.

---

## 5. Combining strings

📚 **Concept**: We often need to combine multiple string columns. Pandas offers two main approaches:

1. **`.str.cat()`** - pandas method for concatenating strings
2. **`+` operator** - simpler, more readable for basic cases

<!-- INSTRUCTOR: Show both methods and discuss when each is appropriate -->

### 5.1 Using `.str.cat()`

In [ ]:
# Concatenate with a separator
df["col_c"].str.cat(df["col_d"], sep="-")

### 5.2 Using the `+` operator

In [ ]:
# Same result using + operator
df["col_c"] + "-" + df["col_d"]

💡 **Which to use?**
- `+` operator: Simpler and more readable for basic concatenation
- `.str.cat()`: Better for advanced cases (handling missing values, joining multiple columns)

For beginners, the `+` operator is usually clearer!

In [ ]:
# Create a new combined column
df["combo_cd"] = df["col_c"] + "-" + df["col_d"]
df

### ✅ Quick Check

Before proceeding to the real-world example, can you:
- [ ] Explain why we need the `.str` accessor?
- [ ] Split a string column into multiple columns?
- [ ] Use `.extract()` to pull out patterns from text?
- [ ] Combine two string columns?

---

## 6. Real-world example: Cleaning Stockholm housing data

📚 **Concept**: Real-world data is messy! Let's apply our string operations to clean actual housing sale data from Stockholm.

We'll tackle common data cleaning tasks:
- Removing unwanted characters ("kr", "sqm")
- Converting text to numbers
- Extracting structured information (floor numbers)

<!-- INSTRUCTOR: This is where students see the practical value of string operations -->

### 6.1 Loading and inspecting the data

In [ ]:
# Load Stockholm housing data
df = pd.read_csv("sthlm_raw.csv")

In [ ]:
# Inspect the structure
df.info()

⚠️ **Notice**: Some columns that should be numeric (`rooms`, `sqm`, `price_sold`) are stored as `object` (text)!

This happens when the data contains non-numeric characters. Let's fix this.

In [ ]:
# Look at a few examples
df.head()

### 6.2 Cleaning the `rooms` column

💡 **The Problem**: The `rooms` column uses commas as decimal separators (European format), but pandas needs dots.

In [ ]:
# Check current data type
df["rooms"].dtype

In [ ]:
# View some example values
df["rooms"].head(10)

In [ ]:
# Replace commas with dots
df["rooms"] = df["rooms"].str.replace(",", ".")

In [ ]:
# Convert to float (numeric)
df["rooms"] = df["rooms"].astype(float)

In [ ]:
# Verify the change
df["rooms"].dtype

✅ **Success!** The `rooms` column is now a proper numeric type.

💡 **Efficiency Note**: We just cleaned 64,000+ rows in milliseconds! This is the power of vectorized operations in pandas.

### 6.3 Cleaning the `sqm` column

💡 **The Problem**: The `sqm` column has both unwanted text ("sqm") and commas instead of dots.

In [ ]:
# View example values
df["sqm"].head()

### 6.4 Method chaining for multiple operations

📚 **Concept**: We can chain multiple string operations together:

`series.str.operation1().str.operation2().method3()`

This is cleaner than doing each step separately.

In [ ]:
# Chain operations: remove "sqm", replace "," with ".", convert to float
df["sqm"] = (
    df["sqm"]
    .str.replace("sqm", "")
    .str.replace(",", ".")
    .astype(float)
)

In [ ]:
# Verify the result
df["sqm"].head()

💡 **Tip**: Breaking method chains across multiple lines makes the code more readable!

### 6.5 Cleaning the `price_sold` column

In [ ]:
# View example values
df["price_sold"].head()

In [ ]:
# Remove "kr" and convert to integer
df["price_sold"] = df["price_sold"].str.replace("kr", "").astype(int)

In [ ]:
# View the cleaned data
df.head()

### 6.6 Extracting floor numbers with regex

📚 **Bonus Challenge**: The `address` column contains floor numbers in patterns like `"2 tr"` (2nd floor). Let's extract them!

⚠️ **Advanced Regex**: This pattern uses lookahead and lookbehind. Don't worry if it seems complex - regex mastery takes time!

In [ ]:
# View address examples
df["address"].head(10)

In [ ]:
# Extract floor numbers
# Pattern: (?<= ) one or more digits (?= tr)
# Meaning: digits that are preceded by a space and followed by " tr"
df["address"].str.extract(r'((?<= )[0-9]+(?= tr))')

💡 **Note**: Not all addresses have floor numbers, so we'll see `NaN` (missing values) for those.

In [ ]:
# Create floor_number column and convert to float
df["floor_number"] = df["address"].str.extract(r'((?<= )[0-9]+(?= tr))').astype(float)

In [ ]:
# See distribution of floor numbers
df["floor_number"].value_counts().sort_index()

✅ **Key Takeaway**: String operations enable us to extract structured information from messy, real-world text data!

---

## Summary

### 1. Core Concepts

**The `.str` Accessor**
- Series don't have string methods by default
- Use `.str` to access string operations: `series.str.method()`
- Operations apply to every value in the column

**Common String Methods**
- `.upper()` / `.lower()` - Change case
- `.split(delimiter, expand=True)` - Split into columns
- `.replace(old, new)` - Replace text
- `.extract(pattern)` - Extract using regex
- `.str.cat()` or `+` - Combine strings

### 2. Essential Skills

✅ **Data Cleaning**
- Remove unwanted characters from columns
- Convert text columns to numeric types
- Replace decimal separators

✅ **Pattern Extraction**
- Use basic regex patterns like `\d+` for digits
- Extract structured data from unstructured text
- Handle missing values when patterns don't match

✅ **Method Chaining**
- Chain multiple operations for cleaner code
- Break chains across lines for readability

### 3. Pro Tips

💡 **Efficiency**
- Assign split results to multiple columns at once
- String operations on 60,000+ rows execute in milliseconds
- Always check data types with `.info()` after cleaning

💡 **Best Practices**
- Use `+` operator for simple string combination
- Break method chains across lines for readability
- Test regex patterns on small samples first

### 4. Learning Resources

📚 **Documentation**
- [Pandas string methods](https://pandas.pydata.org/docs/user_guide/text.html)
- [Regex tutorial for beginners](https://regexone.com/)

### ✅ Final Check

Can you now:
- [ ] Use the `.str` accessor to apply string methods?
- [ ] Split a column and create new columns from the parts?
- [ ] Clean text data by removing unwanted characters?
- [ ] Convert cleaned text columns to numeric types?
- [ ] Extract patterns using basic regex?
- [ ] Combine string columns using `+` or `.str.cat()`?

<!-- INSTRUCTOR: Review the key concepts and introduce the lab exercises -->